# ABC2Vec — Notebook 04: Pre-Training Objectives

Implements all four pre-training objectives from the research brief:

1. **Masked Music Modeling (MMM)** — BERT-style bar masking prediction (baseline, from CLaMP)
2. **Section Contrastive Loss (SCL)** — *Novel*: A-part and B-part of same tune are positives
3. **Transposition Invariance (TI)** — Semi-novel: contrastive loss between transposed versions
4. **Variant-Aware Contrastive (VAC)** — *Novel*: same-tune different-setting as hard positives

Each objective is implemented as a loss function module that can be combined during training.

In [8]:
import sys, json
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

PROJECT_DIR = Path('/Volumes/LLModels/ABC2VEC')
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
sys.path.insert(0, str(PROJECT_DIR))

from abc2vec.tokenizer import ABCVocabulary, BarPatchifier
from abc2vec.model import ABC2VecConfig, ABC2VecModel

## 4.1 Objective 1: Masked Music Modeling (MMM)

Standard BERT-style masked language modeling adapted for ABC notation bars.
Randomly mask bar patches and predict their character content.

This is the baseline objective used in CLaMP.

In [9]:
class MaskedMusicModelingLoss(nn.Module):
    """
    Loss for Masked Music Modeling (MMM).
    
    Cross-entropy loss over predicted characters in masked bars.
    """
    
    def __init__(self, pad_idx: int = 0):
        super().__init__()
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=pad_idx, reduction='mean')
    
    def forward(self, mmm_logits: torch.Tensor, mmm_targets: torch.Tensor,
                mmm_mask: torch.Tensor) -> torch.Tensor:
        """
        Args:
            mmm_logits: (batch, num_masked, max_bar_length, vocab_size)
            mmm_targets: (batch, num_masked, max_bar_length) — ground truth char indices
            mmm_mask: (batch, num_masked) — True for valid masked positions
        
        Returns:
            loss: scalar
        """
        # Only compute loss on valid masked positions
        # Flatten to (N, vocab_size) and (N,)
        valid = mmm_mask.unsqueeze(-1).expand_as(mmm_targets)  # (batch, num_masked, max_bar_length)
        
        logits_flat = mmm_logits[valid].view(-1, mmm_logits.shape[-1])  # (N, vocab_size)
        targets_flat = mmm_targets[valid].view(-1)  # (N,)
        
        if logits_flat.shape[0] == 0:
            return torch.tensor(0.0, device=mmm_logits.device, requires_grad=True)
        
        loss = self.loss_fn(logits_flat, targets_flat)
        return loss

# Test
mmm_loss_fn = MaskedMusicModelingLoss(pad_idx=0)

# Dummy data
batch_size, num_masked, max_bar_len, vocab_size = 4, 5, 64, 128
dummy_logits = torch.randn(batch_size, num_masked, max_bar_len, vocab_size)
dummy_targets = torch.randint(0, vocab_size, (batch_size, num_masked, max_bar_len))
dummy_mask = torch.ones(batch_size, num_masked, dtype=torch.bool)
dummy_mask[0, 3:] = False

loss = mmm_loss_fn(dummy_logits, dummy_targets, dummy_mask)
print(f"MMM Loss: {loss.item():.4f}")
print(f"Expected ~log({vocab_size}) = {np.log(vocab_size):.4f} for random predictions")

MMM Loss: 5.3208
Expected ~log(128) = 4.8520 for random predictions


## 4.2 Objective 2: Section Contrastive Loss (SCL) — NOVEL

Folk tunes have an AABB structure. Train the model such that:
- A-section and B-section of the **same tune** are closer (positive pair)
- Sections from **different tunes** are farther apart (negative pairs)

Uses InfoNCE loss (NT-Xent) over section embeddings.

In [10]:
class SectionContrastiveLoss(nn.Module):
    """
    Section Contrastive Loss (SCL) — Novel objective.
    
    Given a batch of (section_A, section_B) pairs from the same tune,
    trains the model so that paired sections are closer in embedding space
    than sections from different tunes.
    
    Uses symmetric InfoNCE (NT-Xent) loss.
    """
    
    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, emb_a: torch.Tensor, emb_b: torch.Tensor) -> torch.Tensor:
        """
        Args:
            emb_a: (batch, d_embed) — embeddings of A-sections
            emb_b: (batch, d_embed) — embeddings of B-sections (same tune)
        
        Returns:
            loss: scalar
        """
        batch_size = emb_a.shape[0]
        device = emb_a.device
        
        # Normalize embeddings (should already be normalized, but ensure)
        emb_a = F.normalize(emb_a, p=2, dim=-1)
        emb_b = F.normalize(emb_b, p=2, dim=-1)
        
        # Compute similarity matrix: (batch, batch)
        # sim[i, j] = cosine_sim(emb_a[i], emb_b[j])
        sim_ab = torch.matmul(emb_a, emb_b.T) / self.temperature  # (batch, batch)
        sim_ba = sim_ab.T  # (batch, batch)
        
        # Labels: diagonal entries are the positive pairs
        labels = torch.arange(batch_size, device=device)
        
        # Symmetric InfoNCE
        loss_ab = F.cross_entropy(sim_ab, labels)
        loss_ba = F.cross_entropy(sim_ba, labels)
        
        return (loss_ab + loss_ba) / 2

# Test
scl_loss_fn = SectionContrastiveLoss(temperature=0.07)

# Dummy: batch of 8 section pairs
emb_a = F.normalize(torch.randn(8, 128), dim=-1)
emb_b = F.normalize(torch.randn(8, 128), dim=-1)

loss = scl_loss_fn(emb_a, emb_b)
print(f"SCL Loss (random): {loss.item():.4f}")
print(f"Expected ~log({8}) = {np.log(8):.4f} for random embeddings")

# Test with identical pairs (should be low loss)
loss_identical = scl_loss_fn(emb_a, emb_a)
print(f"SCL Loss (identical pairs): {loss_identical.item():.4f}")

SCL Loss (random): 3.7467
Expected ~log(8) = 2.0794 for random embeddings
SCL Loss (identical pairs): 0.0000


## 4.3 Objective 3: Transposition Invariance (TI)

The same tune in different keys should produce similar embeddings.
Contrastive loss between original and transposed versions of the same tune.

In [11]:
class TranspositionInvarianceLoss(nn.Module):
    """
    Transposition Invariance Loss (TI).
    
    Given a batch of (original, transposed) tune pairs,
    trains the model so that transposed versions are close in embedding space.
    
    Uses InfoNCE with in-batch negatives.
    """
    
    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, emb_orig: torch.Tensor, emb_trans: torch.Tensor) -> torch.Tensor:
        """
        Args:
            emb_orig: (batch, d_embed) — embeddings of original tunes
            emb_trans: (batch, d_embed) — embeddings of transposed versions
        
        Returns:
            loss: scalar
        """
        batch_size = emb_orig.shape[0]
        device = emb_orig.device
        
        emb_orig = F.normalize(emb_orig, p=2, dim=-1)
        emb_trans = F.normalize(emb_trans, p=2, dim=-1)
        
        # Concatenate all embeddings for richer negative set
        # emb_all: (2*batch, d_embed)
        emb_all = torch.cat([emb_orig, emb_trans], dim=0)
        
        # Similarity matrix: (2*batch, 2*batch)
        sim = torch.matmul(emb_all, emb_all.T) / self.temperature
        
        # Create labels: positive pairs are (i, i+batch) and (i+batch, i)
        labels = torch.cat([
            torch.arange(batch_size, 2 * batch_size, device=device),  # orig → trans
            torch.arange(0, batch_size, device=device),               # trans → orig
        ])
        
        # Mask out self-similarity (diagonal)
        mask = torch.eye(2 * batch_size, dtype=torch.bool, device=device)
        sim = sim.masked_fill(mask, float('-inf'))
        
        loss = F.cross_entropy(sim, labels)
        return loss

# Test
ti_loss_fn = TranspositionInvarianceLoss(temperature=0.07)

emb_orig = F.normalize(torch.randn(8, 128), dim=-1)
emb_trans = F.normalize(torch.randn(8, 128), dim=-1)

loss = ti_loss_fn(emb_orig, emb_trans)
print(f"TI Loss (random): {loss.item():.4f}")
print(f"Expected ~log({16-1}) = {np.log(15):.4f}")

# Test with near-identical transpositions
emb_trans_close = emb_orig + torch.randn_like(emb_orig) * 0.01
loss_close = ti_loss_fn(emb_orig, F.normalize(emb_trans_close, dim=-1))
print(f"TI Loss (close pairs): {loss_close.item():.4f}")

TI Loss (random): 3.1090
Expected ~log(15) = 2.7081
TI Loss (close pairs): 0.0000


## 4.4 Objective 4: Variant-Aware Contrastive (VAC) — NOVEL

Uses ground-truth variant labels from The Session:
- Same tune, different "setting" (notation variant) = hard positive
- Different tunes = negatives

This directly trains the model for the most important downstream task: tune variant retrieval.

In [12]:
class VariantAwareContrastiveLoss(nn.Module):
    """
    Variant-Aware Contrastive Loss (VAC) — Novel objective.
    
    Given a batch of tune embeddings with group IDs (same group = same tune family),
    uses a contrastive loss where:
    - Positive pairs: tunes from the same family (different settings)
    - Negative pairs: tunes from different families
    
    Supports multiple positives per anchor via supervised contrastive loss (SupCon).
    """
    
    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, embeddings: torch.Tensor, group_ids: torch.Tensor) -> torch.Tensor:
        """
        Args:
            embeddings: (batch, d_embed) — tune embeddings
            group_ids: (batch,) — integer group IDs (same ID = same tune family)
        
        Returns:
            loss: scalar
        """
        batch_size = embeddings.shape[0]
        device = embeddings.device
        
        embeddings = F.normalize(embeddings, p=2, dim=-1)
        
        # Similarity matrix
        sim = torch.matmul(embeddings, embeddings.T) / self.temperature
        
        # Create positive mask: group_ids[i] == group_ids[j]
        ids = group_ids.unsqueeze(1)  # (batch, 1)
        pos_mask = (ids == ids.T).float()  # (batch, batch)
        
        # Remove self from positives
        self_mask = torch.eye(batch_size, device=device)
        pos_mask = pos_mask - self_mask
        
        # Check if there are any positive pairs
        pos_count = pos_mask.sum(dim=1)  # (batch,)
        valid_anchors = pos_count > 0
        
        if not valid_anchors.any():
            return torch.tensor(0.0, device=device, requires_grad=True)
        
        # Mask out self-similarity before log_softmax
        sim = sim.masked_fill(self_mask.bool(), float('-inf'))
        
        # Log-sum-exp for denominator (per row)
        log_denom = torch.logsumexp(sim, dim=1)  # (batch,)
        
        # log p(j|i) = sim(i,j) - log_denom(i)
        log_prob = sim - log_denom.unsqueeze(1)  # (batch, batch)
        
        # Sum log-probs over positive pairs using torch.where to avoid -inf * 0 = nan
        pos_log_prob = torch.where(
            pos_mask.bool(), log_prob, torch.zeros_like(log_prob)
        ).sum(dim=1) / pos_count.clamp(min=1)
        
        # Only include anchors that have at least one positive
        loss = -pos_log_prob[valid_anchors].mean()
        
        return loss

# Test
vac_loss_fn = VariantAwareContrastiveLoss(temperature=0.07)

# 12 tunes from 4 tune families (3 variants each)
embeddings = F.normalize(torch.randn(12, 128), dim=-1)
group_ids = torch.tensor([0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3])

loss = vac_loss_fn(embeddings, group_ids)
print(f"VAC Loss (random, 4 groups x 3): {loss.item():.4f}")
print(f"Expected ~log({12-3}) = {np.log(9):.4f} for random embeddings")

# Test with grouped embeddings (should be lower)
base_embs = torch.randn(4, 128)
grouped_embs = torch.cat([base_embs[i:i+1].expand(3, -1) + torch.randn(3, 128) * 0.05 for i in range(4)])
grouped_embs = F.normalize(grouped_embs, dim=-1)

loss_grouped = vac_loss_fn(grouped_embs, group_ids)
print(f"VAC Loss (grouped, small noise): {loss_grouped.item():.4f}")
assert loss_grouped < loss, "Grouped embeddings should have lower loss"
print("Sanity check passed: grouped < random")

VAC Loss (random, 4 groups x 3): 2.8348
Expected ~log(9) = 2.1972 for random embeddings
VAC Loss (grouped, small noise): 0.6932
Sanity check passed: grouped < random


## 4.5 Combined Multi-Objective Loss

Combines all four objectives with configurable weights.

In [13]:
class ABC2VecLoss(nn.Module):
    """
    Combined multi-objective loss for ABC2Vec pre-training.
    
    L = λ_mmm * L_MMM + λ_scl * L_SCL + λ_ti * L_TI + λ_vac * L_VAC
    """
    
    def __init__(self, config: ABC2VecConfig,
                 lambda_mmm: float = 1.0,
                 lambda_scl: float = 0.5,
                 lambda_ti: float = 0.5,
                 lambda_vac: float = 0.5):
        super().__init__()
        self.lambda_mmm = lambda_mmm
        self.lambda_scl = lambda_scl
        self.lambda_ti = lambda_ti
        self.lambda_vac = lambda_vac
        
        self.mmm_loss = MaskedMusicModelingLoss(pad_idx=config.pad_idx)
        self.scl_loss = SectionContrastiveLoss(temperature=config.temperature)
        self.ti_loss = TranspositionInvarianceLoss(temperature=config.temperature)
        self.vac_loss = VariantAwareContrastiveLoss(temperature=config.temperature)
    
    def forward(self, 
                mmm_logits=None, mmm_targets=None, mmm_mask=None,
                scl_emb_a=None, scl_emb_b=None,
                ti_emb_orig=None, ti_emb_trans=None,
                vac_embeddings=None, vac_group_ids=None) -> dict:
        """
        Compute combined loss. Each component is optional.
        
        Returns dict with individual losses and total loss.
        """
        losses = {}
        total = torch.tensor(0.0, requires_grad=True)
        
        # MMM
        if mmm_logits is not None and mmm_targets is not None and mmm_mask is not None:
            l_mmm = self.mmm_loss(mmm_logits, mmm_targets, mmm_mask)
            losses['mmm'] = l_mmm.item()
            total = total + self.lambda_mmm * l_mmm
        
        # SCL
        if scl_emb_a is not None and scl_emb_b is not None:
            l_scl = self.scl_loss(scl_emb_a, scl_emb_b)
            losses['scl'] = l_scl.item()
            total = total + self.lambda_scl * l_scl
        
        # TI
        if ti_emb_orig is not None and ti_emb_trans is not None:
            l_ti = self.ti_loss(ti_emb_orig, ti_emb_trans)
            losses['ti'] = l_ti.item()
            total = total + self.lambda_ti * l_ti
        
        # VAC
        if vac_embeddings is not None and vac_group_ids is not None:
            l_vac = self.vac_loss(vac_embeddings, vac_group_ids)
            losses['vac'] = l_vac.item()
            total = total + self.lambda_vac * l_vac
        
        losses['total'] = total.item()
        
        return total, losses

# Test combined loss
config = ABC2VecConfig.load(PROCESSED_DIR / 'model_config.json')
combined_loss = ABC2VecLoss(config)

total, loss_dict = combined_loss(
    mmm_logits=dummy_logits, mmm_targets=dummy_targets, mmm_mask=dummy_mask,
    scl_emb_a=emb_a, scl_emb_b=emb_b,
    ti_emb_orig=emb_orig, ti_emb_trans=emb_trans,
    vac_embeddings=grouped_embs, vac_group_ids=group_ids
)

print("Combined Loss:")
for k, v in loss_dict.items():
    print(f"  {k}: {v:.4f}")
print(f"\nGradient exists: {total.requires_grad}")

Combined Loss:
  mmm: 5.3208
  scl: 3.7467
  ti: 3.1090
  vac: 0.6932
  total: 9.0952

Gradient exists: True


## 4.6 Export Losses Module

In [14]:
losses_code = '''
"""Pre-training objectives for ABC2Vec."""

import torch
import torch.nn as nn
import torch.nn.functional as F


class MaskedMusicModelingLoss(nn.Module):
    def __init__(self, pad_idx=0):
        super().__init__()
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=pad_idx, reduction="mean")

    def forward(self, mmm_logits, mmm_targets, mmm_mask):
        valid = mmm_mask.unsqueeze(-1).expand_as(mmm_targets)
        logits_flat = mmm_logits[valid].view(-1, mmm_logits.shape[-1])
        targets_flat = mmm_targets[valid].view(-1)
        if logits_flat.shape[0] == 0:
            return torch.tensor(0.0, device=mmm_logits.device, requires_grad=True)
        return self.loss_fn(logits_flat, targets_flat)


class SectionContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, emb_a, emb_b):
        batch_size = emb_a.shape[0]
        emb_a = F.normalize(emb_a, p=2, dim=-1)
        emb_b = F.normalize(emb_b, p=2, dim=-1)
        sim_ab = torch.matmul(emb_a, emb_b.T) / self.temperature
        labels = torch.arange(batch_size, device=emb_a.device)
        return (F.cross_entropy(sim_ab, labels) + F.cross_entropy(sim_ab.T, labels)) / 2


class TranspositionInvarianceLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, emb_orig, emb_trans):
        batch_size = emb_orig.shape[0]
        device = emb_orig.device
        emb_orig = F.normalize(emb_orig, p=2, dim=-1)
        emb_trans = F.normalize(emb_trans, p=2, dim=-1)
        emb_all = torch.cat([emb_orig, emb_trans], dim=0)
        sim = torch.matmul(emb_all, emb_all.T) / self.temperature
        labels = torch.cat([
            torch.arange(batch_size, 2 * batch_size, device=device),
            torch.arange(0, batch_size, device=device)])
        mask = torch.eye(2 * batch_size, dtype=torch.bool, device=device)
        sim = sim.masked_fill(mask, float("-inf"))
        return F.cross_entropy(sim, labels)


class VariantAwareContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, embeddings, group_ids):
        batch_size = embeddings.shape[0]
        device = embeddings.device
        embeddings = F.normalize(embeddings, p=2, dim=-1)
        sim = torch.matmul(embeddings, embeddings.T) / self.temperature
        group_ids = group_ids.unsqueeze(0)
        pos_mask = (group_ids == group_ids.T).float()
        self_mask = torch.eye(batch_size, device=device)
        pos_mask = pos_mask - self_mask
        pos_count = pos_mask.sum(dim=1)
        valid = pos_count > 0
        if not valid.any():
            return torch.tensor(0.0, device=device, requires_grad=True)
        sim = sim.masked_fill(self_mask.bool(), float("-inf"))
        log_denom = torch.logsumexp(sim, dim=1)
        log_prob = sim - log_denom.unsqueeze(1)
        pos_log_prob = (log_prob * pos_mask).sum(dim=1) / pos_count.clamp(min=1)
        return -pos_log_prob[valid].mean()


class ABC2VecLoss(nn.Module):
    def __init__(self, config, lambda_mmm=1.0, lambda_scl=0.5, lambda_ti=0.5, lambda_vac=0.5):
        super().__init__()
        self.lambda_mmm = lambda_mmm
        self.lambda_scl = lambda_scl
        self.lambda_ti = lambda_ti
        self.lambda_vac = lambda_vac
        self.mmm_loss = MaskedMusicModelingLoss(pad_idx=config.pad_idx)
        self.scl_loss = SectionContrastiveLoss(temperature=config.temperature)
        self.ti_loss = TranspositionInvarianceLoss(temperature=config.temperature)
        self.vac_loss = VariantAwareContrastiveLoss(temperature=config.temperature)

    def forward(self, mmm_logits=None, mmm_targets=None, mmm_mask=None,
                scl_emb_a=None, scl_emb_b=None,
                ti_emb_orig=None, ti_emb_trans=None,
                vac_embeddings=None, vac_group_ids=None):
        losses = {}
        total = torch.tensor(0.0, requires_grad=True)
        if mmm_logits is not None:
            l = self.mmm_loss(mmm_logits, mmm_targets, mmm_mask)
            losses["mmm"] = l.item()
            total = total + self.lambda_mmm * l
        if scl_emb_a is not None:
            l = self.scl_loss(scl_emb_a, scl_emb_b)
            losses["scl"] = l.item()
            total = total + self.lambda_scl * l
        if ti_emb_orig is not None:
            l = self.ti_loss(ti_emb_orig, ti_emb_trans)
            losses["ti"] = l.item()
            total = total + self.lambda_ti * l
        if vac_embeddings is not None:
            l = self.vac_loss(vac_embeddings, vac_group_ids)
            losses["vac"] = l.item()
            total = total + self.lambda_vac * l
        losses["total"] = total.item()
        return total, losses
'''

losses_path = PROJECT_DIR / 'abc2vec' / 'losses.py'
losses_path.write_text(losses_code)
print(f"Losses module saved to {losses_path}")

Losses module saved to /Volumes/LLModels/ABC2VEC/abc2vec/losses.py
